In [0]:
# Install the latest version of the langchain package quietly
%pip install -qU langchain

# Install the latest version of the langchain-openai package quietly
%pip install -qU langchain-openai

# Force reinstall a specific version of numpy package
%pip install --force-reinstall numpy==1.26.2

# Install the latest version of the langchain-community package quietly
%pip install -qU langchain-community

# Install the latest version of the langchain-experimental package quietly
%pip install -qU langchain-experimental

In [0]:
# Restart the Python process to ensure that newly installed libraries are available
dbutils.library.restartPython()

In [0]:
import json  # For parsing and manipulating JSON data
import logging  # For logging information
import pandas as pd  # For data manipulation
import pyspark.sql.functions as F
import time  # For time-related functions
import warnings  # For handling warnings
from datetime import datetime  # For manipulating dates and times
from IPython.display import display, Markdown  # For displaying rich content in notebooks
from langchain.chat_models.azure_openai import AzureChatOpenAI # Import the AzureChatOpenAI class from the langchain.chat_models.azure_openai module
from langchain.agents import AgentExecutor  # For executing agents
from langchain.agents.agent_types import AgentType  # For defining agent types
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent, create_spark_dataframe_agent # For creating pandas dataframe agents
from pyspark.sql.types import *  # For defining Spark data types
from tabulate import tabulate  # For tabulating data

warnings.simplefilter(action='ignore', category=FutureWarning)  # Ignore future warnings

logger = logging.getLogger(__name__)  # Create a logger

logger.propagate = False  # Prevent the logger from propagating messages to the root logger

logger.setLevel(logging.INFO)  # Set the logging level to INFO

stream_handler = logging.StreamHandler()  # Create a stream handler

stream_handler.setLevel(logging.INFO)  # Set the stream handler logging level to INFO

formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')  # Define a log message format

stream_handler.setFormatter(formatter)  # Set the formatter for the stream handler

logger.addHandler(stream_handler)  # Add the stream handler to the logger


In [0]:
# Retrieve the OpenAI API key from Databricks secrets
api_key = dbutils.secrets.get(scope="<yourscope", key="<yourkey>")
azure_endpoint  = dbutils.secrets.get(scope="<yourscope>", key="<youruri")

deployment = "gpt-4o"

# # Initialize the AzureChatOpenAI model with the specified parameters
llm = AzureChatOpenAI(
    azure_deployment=deployment, 
    azure_endpoint=azure_endpoint,  
    openai_api_version="2024-12-01-preview",  
    openai_api_key=api_key,  
    model=deployment,
    temperature=0 
)

In [0]:
# Retrieve the list of catalogs
catalogs_df = spark.sql("SHOW CATALOGS")

# Define catalogs to be excluded
excluded_catalogs = ["hive_metastore", "system", "samples", "main", "__databricks_internal"]

# Filter out the excluded catalogs
catalogs_df = catalogs_df.filter(~catalogs_df.catalog.isin(*excluded_catalogs))

table_list = []

# Iterate over each catalog
for cat_sch in catalogs_df.collect():
    catalog = cat_sch.catalog
    
    # Query to retrieve table information from the current catalog
    query = f"""
    SELECT 
        table_catalog,
        table_schema,
        table_name
    FROM `{catalog}`.information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'data_admin')
    AND table_type <> 'VIEW'
    """
    
    # Execute the query and convert the result to a Pandas DataFrame
    result = spark.sql(query).toPandas()
    
    # Append the result to the table list
    table_list.append(result)

# Concatenate all DataFrames into a single DataFrame
table_list_pdf = pd.concat(table_list)

In [0]:
# Iterate over each row in the table list DataFrame
for index, row in table_list_pdf.iterrows():
    # Extract catalog, schema, and table name from the current row
    table_catalog = row['table_catalog']
    table_schema = row['table_schema']
    table_name = row['table_name']
 

In [0]:
df_list = []

# Iterate over each row in the table list DataFrame
for index, row in table_list_pdf.iterrows():
    table_catalog = row['table_catalog']
    table_schema = row['table_schema']
    table_name = row['table_name']

    logger.info(f"Processing table: {table_catalog}.{table_schema}.{table_name}")

    # Query to select all data from the current table
    table_query = f"""SELECT * FROM {table_catalog}.{table_schema}.{table_name}"""

    # Execute the query and limit the result to 5 rows
    df = spark.sql(table_query).limit(5)

    # Create a Spark DataFrame agent for analysis
    agent = create_spark_dataframe_agent(
        llm,  # Azure OpenAI model initialized in the previous cell
        df,  # DataFrame to be analyzed
        agent_type=AgentType.OPENAI_FUNCTIONS,  # Specify the agent type
        agent_executor_kwargs={"handle_parsing_errors": True},  # Handle parsing errors
        allow_dangerous_code=True  # Allow execution of potentially dangerous code
    )
    
    metadata = []

    # Collect column names and data types
    for types in df.dtypes:
        dtypes = {}
        dtypes["Column_Name"] = types[0]
        dtypes["Data_Type"] = types[1]
        metadata.append(dtypes)

    data_dictionary = []

    # Generate descriptions for each column
    for column in metadata:
        dd_dict = {}
        logger.info(f"Processing column {column['Column_Name']} in table {table_catalog}.{table_schema}.{table_name}")
        prompt = f"""
                You are an expert data analyst. Based on the provided dataset and column metadata column: {column['Column_Name']} data type {column["Data_Type"]} provide a Description: [Provide description]\n\n ALWAYS FINISH THE OUTPUT. Never send partial responses. STRICTLY use the context provided to generate your response. DO NOT GENERATE ANYTHING FROM YOUR PRE-TRAINED MEMORY\n\n
                """
        response = agent.invoke({"input": prompt})
        description = response['output']

        dd_dict["Column_Name"] = column["Column_Name"]
        dd_dict["Data_Type"] = column["Data_Type"]
        dd_dict["Description"] = description
        data_dictionary.append(dd_dict)

    # Convert the data dictionary to a Pandas DataFrame
    pdf = pd.DataFrame(data_dictionary)

    # Insert catalog, schema, and table information
    pdf.insert(0, "catalog", table_catalog)
    pdf.insert(1, "schema", table_schema)
    pdf.insert(2, "table", table_name)

    # Append the DataFrame to the list
    df_list.append(pdf)
    
    # Sleep to avoid hitting rate limits
    time.sleep(60)

# Concatenate all DataFrames into a single DataFrame
final_df = pd.concat(df_list)

In [0]:
# Convert the Pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(final_df)

# Write the Spark DataFrame to a table in overwrite mode with schema overwrite option
spark_df.write\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable("generaldata.data_admin.data_dictionary")